# 11 -- One eval panel for every headline model (plan v5 Step 12f)

Ratings only compare when models faced the same opponents: the reference members' own ratings are uncertain
(+/- ~100), and that shared error cancels only between models rated against the same members. So every headline
model is rated against one **panel** (`configs/eval.yaml`): `random`, `greedy`, `step_20000`, `sf_1320`, and two
model **rungs** that fill the empty stretch between 719 and 1320 -- AlphaZero r109 and SFT 500k @62.5k.

Three stages (choose with `STAGES` from the shell, default `1,2,3`):
1. **Calibrate the rungs** -- each plays the 4 existing members and the other rung, 40 games per pairing; both
   are fitted together with the 4 held fixed, then frozen. AZ r109's model row becomes a reference row.
2. **Rate the headline models on the panel** -- each plays only the members its saved games lack (old games stay
   valid), 20 games per member, then is refitted on all its games.
3. **Consistency check** -- the ratings' predicted head-to-head gaps vs. the observed ones (notebook 08's games).

**Resumable:** every batch of games is saved as it ends; a rerun plays only what is missing. **Needs a GPU pod**
(most games involve an LLM). ~9 h for stages 1-2 on an L4.

In [ ]:
# Setup (RunPod): the repo lives on the volume; artifacts under artifact_root (configs/base.yaml).
import os
_here = os.getcwd()
REPO_DIR = os.path.dirname(_here) if os.path.basename(_here) == "notebooks" else _here
%cd {REPO_DIR}
!git log -1 --oneline
!pip install -q -r requirements.txt

import json
import yaml
import torch
ROOT = yaml.safe_load(open("configs/base.yaml"))["artifact_root"]
from engine.stockfish import ensure_stockfish, SF_VERSION
print("Stockfish:", SF_VERSION, ensure_stockfish())
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none -- this needs a GPU pod"
print("GPU:", GPU)

## Settings
The panel comes from `configs/eval.yaml`. `RUNGS` are its last two members as playable specs (`fields` = the
columns the frozen SFT row keeps, so it still shows on its run's curve). `HEADLINE` = the models rated on the
panel (Derek, Nov 1): SFT 50k @15k, SFT 500k @30k, RL run 4 @r20, AlphaZero r109 without search (its with-search
rating is the rung's own, from stage 1). Step 17's rows are added to the panel from its own notebook.

In [ ]:
from eval.anchors import ResultsTable
from eval.panel import coverage, load_panel
from train.sft import gpu_hours_at

STAGES = {s.strip() for s in os.environ.get("STAGES", "1,2,3").split(",") if s.strip()}
MATCH = os.environ.get("MATCH", "round1")            # the head-to-head round stage 3 checks
eval_cfg = yaml.safe_load(open("configs/eval.yaml"))
PANEL = load_panel()
GAMES, BATCH, RUNG_GAMES = eval_cfg["games_per_opponent"], eval_cfg["batch_games"], eval_cfg["rung_games_per_pair"]
EXISTING = ["random", "greedy", "step_20000", "sf_1320"]

SFT50 = "structured_masked_nall_e10_s0"
SFT500 = "structured_masked_nall_e2_s0_scaled_d12_n500000"
label_cpu_500k = json.load(open(os.path.join(ROOT, "data", "processed", "sft_scaled_d12_n500000", "manifest.json")))["stockfish_cpu_seconds"] / 3600
RUNGS = [
    {"name": "az_b6c64_s0@r109", "kind": "alphazero", "run": "az_b6c64_s0", "step": 109, "simulations": 50},
    {"name": f"{SFT500}@62500", "kind": "sft", "run": SFT500, "step": 62500,
     "fields": {"run": SFT500, "step": 62500, "seed": 0, "label_cpu_hours": label_cpu_500k, "dataset_size": 500000,
                "train_gpu_hours": round(gpu_hours_at(os.path.join(ROOT, "checkpoints", "sft", SFT500), 62500), 4)}},
]
assert [r["name"] for r in RUNGS] == PANEL[4:] and PANEL[:4] == EXISTING, "RUNGS / EXISTING must match configs/eval.yaml"

RL4 = "structured_masked_sft25000_s0_curric120_lr5x"             # RL run 4 (all four runs were flat; one stands for them)
HEADLINE = [
    {"name": f"{SFT50}@15000", "kind": "sft", "run": SFT50, "step": 15000},                      # SFT, 50k data
    {"name": f"{SFT500}@30000", "kind": "sft", "run": SFT500, "step": 30000},                    # SFT, 500k data, 1 pass
    {"name": f"rl_{RL4}@r20", "kind": "llm_rl", "run": RL4, "step": 20},                          # LLM RL
    {"name": "az_b6c64_s0_nosearch@r109", "kind": "alphazero", "run": "az_b6c64_s0", "step": 109,
     "simulations": 0},                                     # AlphaZero r109 without search (with search = the rung)
]

results = ResultsTable(os.path.join(ROOT, eval_cfg["results_file"]))
print("stages:", sorted(STAGES), "| panel:", PANEL)
print("rungs frozen:", {r["name"]: results.role(r["name"]) == "reference" for r in RUNGS})
cov = coverage(results, PANEL)
for s in HEADLINE:
    r = results.rows.get(s["name"])
    print(f"{s['name']:<62} {r['rating'] if r else '-':>7}  saved games per member: {cov.get(s['name'], {})}")

## Stage 1: calibrate and freeze the rungs
~4-5 h (the SFT rung's games need the GPU). Games are saved as `kind = calib` in `games.csv` after every batch;
skipped once both rungs are frozen.

In [ ]:
from eval.anchors import build_reference_agents, close_agents
from eval.elo import format_table
from eval.panel import calibrate_rungs

if "1" in STAGES:
    existing = build_reference_agents(results, EXISTING, ROOT)
    try:
        est = calibrate_rungs(RUNGS, results, ROOT, existing, games_per_pair=RUNG_GAMES, batch_games=BATCH)
    finally:
        close_agents(existing)
    if est:
        print(format_table([est[r["name"]] for r in RUNGS]))
    torch.cuda.empty_cache()
else:
    print("stage 1 skipped")

## Stage 2: rate the headline models on the full panel
~4-5 h. Each model plays only the members its saved games lack (20 games each), then is refitted on all its
games; its note keeps the old rating. Fails at once if the rungs aren't frozen yet (stage 1 first).

In [ ]:
from eval.panel import extend

if "2" in STAGES:
    results = ResultsTable(results.path)
    opponents = build_reference_agents(results, PANEL, ROOT)
    try:
        for s in HEADLINE:
            print(f"\n== {s['name']}")
            extend(s, results, ROOT, opponents, games_per_opponent=GAMES, batch_games=BATCH, gpu=GPU)
            torch.cuda.empty_cache()
    finally:
        close_agents(opponents)
else:
    print("stage 2 skipped")

## Stage 3: the panel table and the consistency check
Only comparable ratings are used: the panel members and the models with all their panel games. For each
head-to-head pair where both players have one, `predicted_diff` is the gap the ratings predict and `consistent`
says whether it falls inside the observed gap's 95% CI. Inconsistent pairs mean the panel can't be trusted yet.

In [ ]:
import pandas as pd
from eval.match import head_to_head_table
from eval.panel import panel_ratings

if "3" in STAGES:
    t = ResultsTable(results.path)
    R = panel_ratings(t, PANEL, GAMES)
    cols = ["name", "role", "rating", "ci_low", "ci_high", "games", "wins", "draws", "losses", "train_gpu_hours", "sec_per_move"]
    display(pd.DataFrame([t.rows[n] for n in R])[cols].assign(rating=lambda d: d.rating.astype(float))
            .sort_values("rating", ascending=False))
    off = [s["name"] for s in HEADLINE if s["name"] not in R]
    print("headline rows not yet on the full panel:", off or "none")
    games = [g for g in t.games(kind="h2h") if g["subject"].startswith(MATCH + "|")]
    table = pd.DataFrame(head_to_head_table(games, ratings=R))
    display(table[["a", "b", "games", "a_wins", "draws", "a_losses", "elo_diff", "ci_low", "ci_high",
                   "predicted_diff", "predicted_score", "consistent"]])
else:
    print("stage 3 skipped")